<a href="https://colab.research.google.com/github/2403A51L41/ReinforcementLearning/blob/main/RL_Assign_11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
{
  "nbformat": 4,
  "nbformat_minor": 0,
  "metadata": {
    "colab": {
      "provenance": []
    },
    "kernelspec": {
      "name": "python3",
      "display_name": "Python 3"
    },
    "language_info": {
      "name": "python"
    }
  },
  "cells": [
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {
        "id": "W9k0iFNxqVi5"
      },
      "outputs": [],
      "source": [
        "gamma = 0.9\n",
        "theta = 1e-10\n",
        "\n",
        "states = [\"S0\", \"S1\", \"G\"]\n",
        "non_terminal_states = [\"S0\", \"S1\"]\n",
        "actions = [\"Up\", \"Down\", \"Left\", \"Right\"]\n",
        "\n",
        "# Grid World\n",
        "# S0   S1\n",
        "# WALL G\n",
        "\n",
        "grid = [\n",
        "    [\"S0\", \"S1\"],\n",
        "    [\"WALL\", \"G\"]\n",
        "]\n",
        "\n",
        "positions = {\n",
        "    \"S0\": (0, 0),\n",
        "    \"S1\": (0, 1),\n",
        "    \"G\": (1, 1)\n",
        "}\n",
        "\n",
        "wall = (1, 0)\n",
        "\n",
        "deltas = {\n",
        "    \"Up\": (-1, 0),\n",
        "    \"Down\": (1, 0),\n",
        "    \"Left\": (0, -1),\n",
        "    \"Right\": (0, 1)\n",
        "}\n",
        "\n",
        "# Define transitions and rewards\n",
        "transition = {}\n",
        "reward = {}\n",
        "\n",
        "for s in non_terminal_states:\n",
        "    r, c = positions[s]\n",
        "\n",
        "    for a in actions:\n",
        "        dr, dc = deltas[a]\n",
        "        nr, nc = r + dr, c + dc\n",
        "\n",
        "        if not (0 <= nr < 2 and 0 <= nc < 2) or (nr, nc) == wall:\n",
        "            next_state = s\n",
        "            immediate_reward = -1\n",
        "        else:\n",
        "            next_state = grid[nr][nc]\n",
        "            immediate_reward = 100 if next_state == \"G\" else -1\n",
        "\n",
        "        transition[(s, a)] = next_state\n",
        "        reward[(s, a)] = immediate_reward\n",
        "\n",
        "# Display MDP information\n",
        "print(\"STATES:\", states)\n",
        "print(\"ACTIONS:\", actions)\n",
        "print(\"Discount factor:\", gamma)\n",
        "print(\"Normal reward: -1\")\n",
        "print(\"Wall/boundary reward: -1\")\n",
        "print(\"Goal reward: +100\")\n",
        "print(\"Terminal state value: V(G) = 0\")\n",
        "\n",
        "print(\"\\nTRANSITION MAPPING\")\n",
        "for s in non_terminal_states:\n",
        "    for a in actions:\n",
        "        print(\n",
        "            f\"State={s}, Action={a}, \"\n",
        "            f\"Next State={transition[(s, a)]}, \"\n",
        "            f\"Reward={reward[(s, a)]}\"\n",
        "        )\n",
        "\n",
        "# Initialize policy and state values\n",
        "policy = {\n",
        "    \"S0\": \"Up\",\n",
        "    \"S1\": \"Up\"\n",
        "}\n",
        "\n",
        "V = {s: 0.0 for s in states}\n",
        "\n",
        "print(\"\\nINITIAL POLICY:\", policy)\n",
        "print(\"INITIAL VALUES:\", V)\n",
        "\n",
        "# Policy Iteration\n",
        "policy_iteration = 0\n",
        "\n",
        "while True:\n",
        "    policy_iteration += 1\n",
        "\n",
        "    print(f\"\\n===== POLICY ITERATION {policy_iteration} =====\")\n",
        "    print(\"Current Policy:\", policy)\n",
        "\n",
        "    # 1. Policy Evaluation\n",
        "    evaluation_iteration = 0\n",
        "\n",
        "    while True:\n",
        "        new_V = V.copy()\n",
        "\n",
        "        for s in non_terminal_states:\n",
        "            a = policy[s]\n",
        "            next_state = transition[(s, a)]\n",
        "\n",
        "            new_V[s] = (\n",
        "                reward[(s, a)] + gamma * V[next_state]\n",
        "            )\n",
        "\n",
        "        new_V[\"G\"] = 0.0\n",
        "\n",
        "        delta = max(abs(new_V[s] - V[s]) for s in states)\n",
        "        V = new_V\n",
        "        evaluation_iteration += 1\n",
        "\n",
        "        print(\n",
        "            f\"Evaluation {evaluation_iteration}: \"\n",
        "            f\"V(S0)={V['S0']:.6f}, \"\n",
        "            f\"V(S1)={V['S1']:.6f}, \"\n",
        "            f\"V(G)={V['G']:.6f}, \"\n",
        "            f\"Delta={delta:.6f}\"\n",
        "        )\n",
        "\n",
        "        if delta < theta:\n",
        "            break\n",
        "\n",
        "    # 2. Calculate Q values\n",
        "    Q = {}\n",
        "\n",
        "    print(\"\\nACTION VALUES Q_pi(s, a)\")\n",
        "\n",
        "    for s in non_terminal_states:\n",
        "        for a in actions:\n",
        "            next_state = transition[(s, a)]\n",
        "\n",
        "            Q[(s, a)] = (\n",
        "                reward[(s, a)] + gamma * V[next_state]\n",
        "            )\n",
        "\n",
        "            print(f\"Q({s}, {a}) = {Q[(s, a)]:.6f}\")\n",
        "\n",
        "    # 3. Policy Improvement\n",
        "    improved_policy = {}\n",
        "\n",
        "    for s in non_terminal_states:\n",
        "        max_q = max(Q[(s, a)] for a in actions)\n",
        "\n",
        "        best_actions = [\n",
        "            a for a in actions\n",
        "            if abs(Q[(s, a)] - max_q) < 1e-9\n",
        "        ]\n",
        "\n",
        "        # Keep the current action if it is tied for best.\n",
        "        if policy[s] in best_actions:\n",
        "            improved_policy[s] = policy[s]\n",
        "        else:\n",
        "            improved_policy[s] = best_actions[0]\n",
        "\n",
        "    print(\"\\nImproved Policy:\", improved_policy)\n",
        "\n",
        "    # 4. Check policy stability\n",
        "    if improved_policy == policy:\n",
        "        print(\"Policy is stable.\")\n",
        "        break\n",
        "\n",
        "    policy = improved_policy\n",
        "\n",
        "# Final state values\n",
        "print(\"\\n===== FINAL OPTIMAL STATE VALUES =====\")\n",
        "\n",
        "for s in states:\n",
        "    print(f\"V*({s}) = {V[s]:.6f}\")\n",
        "\n",
        "# Final optimal policy\n",
        "print(\"\\n===== OPTIMAL POLICY =====\")\n",
        "\n",
        "for s in non_terminal_states:\n",
        "    print(f\"Policy({s}) = {policy[s]}\")\n",
        "\n",
        "# Final action-value table\n",
        "print(\"\\n===== FINAL ACTION-VALUE TABLE =====\")\n",
        "print(f\"{'State':<8}{'Up':>12}{'Down':>12}{'Left':>12}{'Right':>12}\")\n",
        "\n",
        "for s in non_terminal_states:\n",
        "    print(\n",
        "        f\"{s:<8}\"\n",
        "        f\"{Q[(s, 'Up')]:>12.4f}\"\n",
        "        f\"{Q[(s, 'Down')]:>12.4f}\"\n",
        "        f\"{Q[(s, 'Left')]:>12.4f}\"\n",
        "        f\"{Q[(s, 'Right')]:>12.4f}\"\n",
        "    )\n",
        "\n",
        "# Bellman Optimality verification\n",
        "print(\"\\n===== BELLMAN OPTIMALITY VERIFICATION =====\")\n",
        "\n",
        "for s in non_terminal_states:\n",
        "    max_q = max(Q[(s, a)] for a in actions)\n",
        "    difference = abs(V[s] - max_q)\n",
        "\n",
        "    print(\n",
        "        f\"State={s}, V*={V[s]:.6f}, \"\n",
        "        f\"max Q*={max_q:.6f}, \"\n",
        "        f\"Difference={difference:.10f}\"\n",
        "    )\n",
        "\n",
        "    assert difference < 1e-8\n",
        "\n",
        "print(\"\\nBellman Optimality verification passed.\")"
      ]
    }
  ]
}